# JPEG Compression Pipeline

This notebook implements the complete JPEG compression pipeline as shown in the diagram:

**Compression Steps:**
1. RGB → YCbCr conversion
2. Down sample color (chroma subsampling)
3. Partition into 8×8 blocks
4. Forward DCT (FDCT)
5. Quantization
6. Entropy encoding (simplified)

**Decompression Steps:**
1. Entropy decoding
2. De-quantization
3. Inverse DCT (IDCT)
4. Up sample color
5. YCbCr → RGB conversion

## Import Required Libraries

In [1]:
import cv2
import numpy as np
from scipy.fftpack import dct, idct
import matplotlib.pyplot as plt
from typing import Tuple, List
import math

# Set up matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

## Step 1: RGB to YCbCr Conversion

In [2]:
def rgb_to_ycbcr(img_rgb: np.ndarray) -> np.ndarray:
    """
    Convert RGB image to YCbCr color space
    Y: Luminance (brightness)
    Cb: Blue difference
    Cr: Red difference
    """
    # Standard conversion matrix
    transform_matrix = np.array([
        [0.299, 0.587, 0.114],
        [-0.168736, -0.331264, 0.5],
        [0.5, -0.418688, -0.081312]
    ])
    
    # Add offset for Cb and Cr channels
    offset = np.array([0, 128, 128])
    
    # Reshape image for matrix multiplication
    h, w, c = img_rgb.shape
    img_flat = img_rgb.reshape(-1, 3).astype(np.float32)
    
    # Apply transformation
    ycbcr_flat = np.dot(img_flat, transform_matrix.T) + offset
    
    # Reshape back to original dimensions
    img_ycbcr = ycbcr_flat.reshape(h, w, 3)
    
    return img_ycbcr.astype(np.float32)

## Step 2: Chroma Subsampling

In [3]:
def chroma_subsample(ycbcr_img: np.ndarray, ratio: str = '4:2:0') -> Tuple[np.ndarray, List[float]]:
    """
    Downsample chroma channels (Cb, Cr) using specified ratio
    Common ratios: '4:4:4', '4:2:2', '4:2:0'
    """
    h, w, c = ycbcr_img.shape
    
    if ratio == '4:4:4':
        # No downsampling
        return ycbcr_img, [1.0, 1.0]
    
    elif ratio == '4:2:2':
        # Downsample horizontally by 2
        y_channel = ycbcr_img[:, :, 0]
        cb_channel = ycbcr_img[:, ::2, 1]
        cr_channel = ycbcbcr_img[:, ::2, 2]
        
        return np.stack([y_channel, cb_channel, cr_channel], axis=2), [1.0, 2.0]
    
    elif ratio == '4:2:0':
        # Downsample both horizontally and vertically by 2
        y_channel = ycbcr_img[:, :, 0]
        
        # Simple averaging for downsampling
        cb_down = cv2.resize(ycbcr_img[:, :, 1], (w//2, h//2), interpolation=cv2.INTER_AREA)
        cr_down = cv2.resize(ycbcr_img[:, :, 2], (w//2, h//2), interpolation=cv2.INTER_AREA)
        
        return np.stack([y_channel, cb_down, cr_down], axis=2), [2.0, 2.0]
    
    else:
        raise ValueError(f"Unsupported chroma subsampling ratio: {ratio}")

## Step 3: Partition into 8×8 Blocks

In [4]:
def split_into_blocks(img: np.ndarray, block_size: int = 8) -> np.ndarray:
    """
    Split image into non-overlapping blocks of specified size
    """
    h, w = img.shape[:2]
    
    # Calculate number of blocks (pad if necessary)
    h_blocks = (h + block_size - 1) // block_size
    w_blocks = (w + block_size - 1) // block_size
    
    # Pad image to make it divisible by block_size
    h_pad = h_blocks * block_size
    w_pad = w_blocks * block_size
    
    padded_img = np.zeros((h_pad, w_pad, 3) if len(img.shape) == 3 else (h_pad, w_pad), 
                          dtype=img.dtype)
    padded_img[:h, :w] = img
    
    # Split into blocks
    if len(img.shape) == 3:  # Color image
        blocks = []
        for c in range(3):
            channel_blocks = []
            for i in range(0, h_pad, block_size):
                for j in range(0, w_pad, block_size):
                    block = padded_img[i:i+block_size, j:j+block_size, c]
                    channel_blocks.append(block)
            blocks.append(channel_blocks)
        return np.array(blocks)
    else:  # Grayscale
        blocks = []
        for i in range(0, h_pad, block_size):
            for j in range(0, w_pad, block_size):
                block = padded_img[i:i+block_size, j:j+block_size]
                blocks.append(block)
        return np.array(blocks)

## Step 4: Forward DCT (Discrete Cosine Transform)

In [5]:
def apply_dct_to_blocks(blocks: np.ndarray) -> np.ndarray:
    """
    Apply 2D Discrete Cosine Transform to each block
    """
    dct_blocks = np.zeros_like(blocks, dtype=np.float32)
    
    if len(blocks.shape) == 3:  # Single channel
        for i, block in enumerate(blocks):
            # Shift values to center around 0 (subtract 128 for JPEG standard)
            block_centered = block.astype(np.float32) - 128
            dct_blocks[i] = dct(dct(block_centered.T, norm='ortho').T, norm='ortho')
    
    else:  # Multiple channels
        for c in range(blocks.shape[0]):
            for i in range(blocks.shape[1]):
                block = blocks[c, i]
                block_centered = block.astype(np.float32) - 128
                dct_blocks[c, i] = dct(dct(block_centered.T, norm='ortho').T, norm='ortho')
    
    return dct_blocks

## Step 5: Quantization

In [6]:
def quantize_blocks(dct_blocks: np.ndarray, quality: int = 50) -> Tuple[np.ndarray, np.ndarray]:
    """
    Quantize DCT coefficients using JPEG standard quantization tables
    Higher quality = less compression, lower quality = more compression
    """
    # Standard JPEG luminance quantization table (for Y channel)
    luminance_quant_table = np.array([
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99]
    ], dtype=np.float32)
    
    # Standard JPEG chrominance quantization table (for Cb and Cr channels)
    chrominance_quant_table = np.array([
        [17, 18, 24, 47, 99, 99, 99, 99],
        [18, 21, 26, 66, 99, 99, 99, 99],
        [24, 26, 56, 99, 99, 99, 99, 99],
        [47, 66, 99, 99, 99, 99, 99, 99],
        [99, 99, 99, 99, 99, 99, 99, 99],
        [99, 99, 99, 99, 99, 99, 99, 99],
        [99, 99, 99, 99, 99, 99, 99, 99],
        [99, 99, 99, 99, 99, 99, 99, 99]
    ], dtype=np.float32)
    
    # Adjust quantization tables based on quality factor (1-100)
    if quality < 50:
        scale_factor = 5000 / quality
    else:
        scale_factor = 200 - quality * 2
    
    scale_factor = scale_factor / 100.0
    scale_factor = max(0.1, min(scale_factor, 10.0))
    
    q_lum = np.clip(np.round(luminance_quant_table * scale_factor), 1, 255)
    q_chrom = np.clip(np.round(chrominance_quant_table * scale_factor), 1, 255)
    
    # Apply quantization
    quantized_blocks = np.zeros_like(dct_blocks)
    
    if len(dct_blocks.shape) == 3:  # Single channel (grayscale)
        for i, block in enumerate(dct_blocks):
            quantized_blocks[i] = np.round(block / q_lum)
    
    else:  # Three channels (Y, Cb, Cr)
        # Y channel
        for i in range(dct_blocks.shape[1]):
            quantized_blocks[0, i] = np.round(dct_blocks[0, i] / q_lum)
        
        # Cb and Cr channels
        for c in range(1, 3):
            for i in range(dct_blocks.shape[1]):
                quantized_blocks[c, i] = np.round(dct_blocks[c, i] / q_chrom)
    
    return quantized_blocks, (q_lum, q_chrom)

## Step 6: Zigzag Scan and Run-length Encoding (Simplified Entropy Encoding)

In [7]:
def zigzag_scan(block: np.ndarray) -> List[float]:
    """
    Perform zigzag scan on an 8x8 block
    """
    zigzag_order = [
        [0, 0], [0, 1], [1, 0], [2, 0], [1, 1], [0, 2], [0, 3], [1, 2],
        [2, 1], [3, 0], [4, 0], [3, 1], [2, 2], [1, 3], [0, 4], [0, 5],
        [1, 4], [2, 3], [3, 2], [4, 1], [5, 0], [6, 0], [5, 1], [4, 2],
        [3, 3], [2, 4], [1, 5], [0, 6], [0, 7], [1, 6], [2, 5], [3, 4],
        [4, 3], [5, 2], [6, 1], [7, 0], [7, 1], [6, 2], [5, 3], [4, 4],
        [3, 5], [2, 6], [1, 7], [2, 7], [3, 6], [4, 5], [5, 4], [6, 3],
        [7, 2], [7, 3], [6, 4], [5, 5], [4, 6], [3, 7], [4, 7], [5, 6],
        [6, 5], [7, 4], [7, 5], [6, 6], [5, 7], [6, 7], [7, 6], [7, 7]
    ]
    
    return [block[y, x] for y, x in zigzag_order]

def run_length_encode(data: List[float]) -> List[Tuple[int, float]]:
    """
    Simple run-length encoding for zigzag-scanned DCT coefficients
    """
    encoded = []
    zero_count = 0
    
    for value in data:
        if value == 0:
            zero_count += 1
        else:
            encoded.append((zero_count, value))
            zero_count = 0
    
    # Add end of block marker
    if zero_count > 0:
        encoded.append((0, 0))  # EOB marker
    
    return encoded

## Complete JPEG Compression Pipeline

In [8]:
def jpeg_compress(img: np.ndarray, quality: int = 75, chroma_subsample_ratio: str = '4:2:0') -> dict:
    """
    Complete JPEG compression pipeline
    Returns dictionary containing compressed data and necessary parameters for decompression
    """
    print(f"Starting JPEG compression with quality={quality}, chroma subsampling={chroma_subsample_ratio}")
    
    # Step 1: RGB to YCbCr
    print("Step 1: Converting RGB to YCbCr...")
    img_ycbcr = rgb_to_ycbcr(img)
    
    # Step 2: Chroma subsampling
    print("Step 2: Applying chroma subsampling...")
    img_subsampled, subsample_factors = chroma_subsample(img_ycbcr, chroma_subsample_ratio)
    
    # Step 3: Split into 8x8 blocks
    print("Step 3: Splitting into 8x8 blocks...")
    blocks = split_into_blocks(img_subsampled)
    
    # Step 4: Apply DCT
    print("Step 4: Applying Forward DCT...")
    dct_blocks = apply_dct_to_blocks(blocks)
    
    # Step 5: Quantization
    print("Step 5: Quantizing DCT coefficients...")
    quantized_blocks, quant_tables = quantize_blocks(dct_blocks, quality)
    
    # Step 6: Zigzag scan and RLE (for demonstration)
    print("Step 6: Applying zigzag scan and RLE...")
    compressed_data = []
    if len(quantized_blocks.shape) == 3:  # Grayscale
        for i, block in enumerate(quantized_blocks):
            zigzag = zigzag_scan(block)
            rle = run_length_encode(zigzag)
            compressed_data.append(rle)
    else:  # Color
        for c in range(quantized_blocks.shape[0]):
            channel_data = []
            for i in range(quantized_blocks.shape[1]):
                zigzag = zigzag_scan(quantized_blocks[c, i])
                rle = run_length_encode(zigzag)
                channel_data.append(rle)
            compressed_data.append(channel_data)
    
    # Store compression metadata
    metadata = {
        'original_shape': img.shape,
        'quality': quality,
        'chroma_subsample_ratio': chroma_subsample_ratio,
        'subsample_factors': subsample_factors,
        'quant_tables': quant_tables,
        'block_size': 8,
        'compressed_data': compressed_data
    }
    
    print("JPEG compression completed!")
    return metadata

## Decompression Functions

In [9]:
def run_length_decode(rle_data: List[Tuple[int, float]], block_size: int = 8) -> np.ndarray:
    """
    Decode run-length encoded data back to zigzag order
    """
    # Create zigzag order mapping
    zigzag_order = [
        [0, 0], [0, 1], [1, 0], [2, 0], [1, 1], [0, 2], [0, 3], [1, 2],
        [2, 1], [3, 0], [4, 0], [3, 1], [2, 2], [1, 3], [0, 4], [0, 5],
        [1, 4], [2, 3], [3, 2], [4, 1], [5, 0], [6, 0], [5, 1], [4, 2],
        [3, 3], [2, 4], [1, 5], [0, 6], [0, 7], [1, 6], [2, 5], [3, 4],
        [4, 3], [5, 2], [6, 1], [7, 0], [7, 1], [6, 2], [5, 3], [4, 4],
        [3, 5], [2, 6], [1, 7], [2, 7], [3, 6], [4, 5], [5, 4], [6, 3],
        [7, 2], [7, 3], [6, 4], [5, 5], [4, 6], [3, 7], [4, 7], [5, 6],
        [6, 5], [7, 4], [7, 5], [6, 6], [5, 7], [6, 7], [7, 6], [7, 7]
    ]
    
    # Reconstruct zigzag array
    zigzag_array = []
    for zero_count, value in rle_data:
        if zero_count == 0 and value == 0:  # EOB marker
            zigzag_array.extend([0] * (block_size * block_size - len(zigzag_array)))
            break
        zigzag_array.extend([0] * zero_count)
        zigzag_array.append(value)
    
    # Fill remaining with zeros if needed
    if len(zigzag_array) < block_size * block_size:
        zigzag_array.extend([0] * (block_size * block_size - len(zigzag_array)))
    
    # Convert zigzag to 2D block
    block = np.zeros((block_size, block_size), dtype=np.float32)
    for idx, (y, x) in enumerate(zigzag_order):
        if idx < len(zigzag_array):
            block[y, x] = zigzag_array[idx]
    
    return block

def dequantize_blocks(quantized_blocks: np.ndarray, quant_tables: Tuple[np.ndarray, np.ndarray]) -> np.ndarray:
    """
    Reverse the quantization process
    """
    q_lum, q_chrom = quant_tables
    dct_blocks = np.zeros_like(quantized_blocks, dtype=np.float32)
    
    if len(quantized_blocks.shape) == 3:  # Single channel
        for i, block in enumerate(quantized_blocks):
            dct_blocks[i] = block * q_lum
    
    else:  # Three channels
        # Y channel
        for i in range(quantized_blocks.shape[1]):
            dct_blocks[0, i] = quantized_blocks[0, i] * q_lum
        
        # Cb and Cr channels
        for c in range(1, 3):
            for i in range(quantized_blocks.shape[1]):
                dct_blocks[c, i] = quantized_blocks[c, i] * q_chrom
    
    return dct_blocks

def apply_idct_to_blocks(dct_blocks: np.ndarray) -> np.ndarray:
    """
    Apply Inverse DCT to each block
    """
    idct_blocks = np.zeros_like(dct_blocks, dtype=np.float32)
    
    if len(dct_blocks.shape) == 3:  # Single channel
        for i, block in enumerate(dct_blocks):
            idct_blocks[i] = idct(idct(block.T, norm='ortho').T, norm='ortho') + 128
    
    else:  # Multiple channels
        for c in range(dct_blocks.shape[0]):
            for i in range(dct_blocks.shape[1]):
                block = dct_blocks[c, i]
                idct_blocks[c, i] = idct(idct(block.T, norm='ortho').T, norm='ortho') + 128
    
    return idct_blocks

def merge_blocks_to_image(blocks: np.ndarray, original_shape: Tuple[int, int], block_size: int = 8) -> np.ndarray:
    """
    Reconstruct image from blocks
    """
    h, w = original_shape[:2]
    
    if len(blocks.shape) == 3:  # Single channel
        num_blocks_h = (h + block_size - 1) // block_size
        num_blocks_w = (w + block_size - 1) // block_size
        
        reconstructed = np.zeros((num_blocks_h * block_size, num_blocks_w * block_size), dtype=np.float32)
        
        for idx, block in enumerate(blocks):
            i = idx // num_blocks_w
            j = idx % num_blocks_w
            reconstructed[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = block
        
        return reconstructed[:h, :w]
    
    else:  # Three channels
        num_blocks_h = (h + block_size - 1) // block_size
        num_blocks_w = (w + block_size - 1) // block_size
        
        reconstructed = np.zeros((num_blocks_h * block_size, num_blocks_w * block_size, 3), dtype=np.float32)
        
        for c in range(blocks.shape[0]):
            channel_blocks = blocks[c]
            channel_reconstructed = np.zeros((num_blocks_h * block_size, num_blocks_w * block_size), dtype=np.float32)
            
            for idx, block in enumerate(channel_blocks):
                i = idx // num_blocks_w
                j = idx % num_blocks_w
                channel_reconstructed[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = block
            
            reconstructed[:, :, c] = channel_reconstructed
        
        return reconstructed[:h, :w, :]

def chroma_upsample(ycbcr_img: np.ndarray, subsample_factors: List[float], original_shape: Tuple[int, int]) -> np.ndarray:
    """
    Upsample chroma channels to original resolution
    """
    h, w = original_shape[:2]
    
    if subsample_factors == [1.0, 1.0]:
        return ycbcr_img
    
    y_channel = ycbcr_img[:, :, 0]
    
    if len(ycbcr_img.shape) == 3 and ycbcr_img.shape[2] == 3:
        cb_channel = ycbcr_img[:, :, 1]
        cr_channel = ycbcr_img[:, :, 2]
        
        # Upsample chroma channels
        cb_upsampled = cv2.resize(cb_channel, (w, h), interpolation=cv2.INTER_LINEAR)
        cr_upsampled = cv2.resize(cr_channel, (w, h), interpolation=cv2.INTER_LINEAR)
        
        return np.stack([y_channel, cb_upsampled, cr_upsampled], axis=2)
    else:
        return ycbcr_img

def ycbcr_to_rgb(img_ycbcr: np.ndarray) -> np.ndarray:
    """
    Convert YCbCr image back to RGB color space
    """
    # Inverse transformation matrix
    transform_matrix = np.array([
        [1.0, 0.0, 1.402],
        [1.0, -0.344136, -0.714136],
        [1.0, 1.772, 0.0]
    ])
    
    # Remove offset
    offset = np.array([0, 128, 128])
    
    # Reshape image for matrix multiplication
    h, w, c = img_ycbcr.shape
    img_flat = img_ycbcr.reshape(-1, 3).astype(np.float32)
    
    # Apply inverse transformation
    img_flat = img_flat - offset
    rgb_flat = np.dot(img_flat, transform_matrix.T)
    
    # Clip values to valid range
    rgb_flat = np.clip(rgb_flat, 0, 255)
    
    # Reshape back to original dimensions
    img_rgb = rgb_flat.reshape(h, w, 3)
    
    return img_rgb.astype(np.uint8)

## Complete JPEG Decompression Pipeline

In [10]:
def jpeg_decompress(metadata: dict) -> np.ndarray:
    """
    Complete JPEG decompression pipeline
    """
    print("Starting JPEG decompression...")
    
    # Extract metadata
    original_shape = metadata['original_shape']
    block_size = metadata['block_size']
    quant_tables = metadata['quant_tables']
    subsample_factors = metadata['subsample_factors']
    compressed_data = metadata['compressed_data']
    
    # Step 1: RLE decode and reconstruct quantized blocks
    print("Step 1: RLE decoding...")
    if isinstance(compressed_data[0][0], tuple):  # Single channel
        num_blocks = len(compressed_data)
        quantized_blocks = np.zeros((num_blocks, block_size, block_size), dtype=np.float32)
        
        for i, rle_data in enumerate(compressed_data):
            quantized_blocks[i] = run_length_decode(rle_data, block_size)
    
    else:  # Three channels
        num_blocks = len(compressed_data[0])
        quantized_blocks = np.zeros((3, num_blocks, block_size, block_size), dtype=np.float32)
        
        for c in range(3):
            for i, rle_data in enumerate(compressed_data[c]):
                quantized_blocks[c, i] = run_length_decode(rle_data, block_size)
    
    # Step 2: Dequantize
    print("Step 2: Dequantizing...")
    dct_blocks = dequantize_blocks(quantized_blocks, quant_tables)
    
    # Step 3: Apply Inverse DCT
    print("Step 3: Applying Inverse DCT...")
    idct_blocks = apply_idct_to_blocks(dct_blocks)
    
    # Step 4: Merge blocks into image
    print("Step 4: Merging blocks into image...")
    img_ycbcr_reconstructed = merge_blocks_to_image(idct_blocks, original_shape, block_size)
    
    # Step 5: Chroma upsampling
    print("Step 5: Chroma upsampling...")
    img_ycbcr_upsampled = chroma_upsample(img_ycbcr_reconstructed, subsample_factors, original_shape)
    
    # Step 6: Convert YCbCr to RGB
    print("Step 6: Converting YCbCr to RGB...")
    img_rgb_reconstructed = ycbcr_to_rgb(img_ycbcr_upsampled)
    
    print("JPEG decompression completed!")
    return img_rgb_reconstructed

## Visualization and Quality Metrics

In [11]:
def calculate_compression_ratio(original_img: np.ndarray, metadata: dict) -> float:
    """
    Calculate approximate compression ratio
    """
    # Original image size in bytes
    original_size = original_img.size * original_img.itemsize
    
    # Estimate compressed size (simplified)
    compressed_data = metadata['compressed_data']
    compressed_size = 0
    
    if isinstance(compressed_data[0][0], tuple):  # Single channel
        for rle_data in compressed_data:
            compressed_size += len(rle_data) * (4 + 8)  # 4 bytes for zero_count, 8 bytes for value
    else:  # Three channels
        for c in range(3):
            for rle_data in compressed_data[c]:
                compressed_size += len(rle_data) * (4 + 8)
    
    # Add metadata size
    compressed_size += 1000  # Approximate size of metadata
    
    return original_size / max(compressed_size, 1)

def calculate_psnr(original: np.ndarray, compressed: np.ndarray) -> float:
    """
    Calculate Peak Signal-to-Noise Ratio (PSNR)
    """
    mse = np.mean((original - compressed) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

## Main Execution - Using Your Image Variable `img`

In [12]:
# Check if 'img' variable exists in the notebook
try:
    # Display the original image
    plt.figure(figsize=(10, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Original Image\nShape: {img.shape}")
    plt.axis('off')
    
    # Show image info
    print(f"Image shape: {img.shape}")
    print(f"Image dtype: {img.dtype}")
    print(f"Image size: {img.size} pixels")
    print(f"Image memory: {img.size * img.itemsize / 1024:.2f} KB")
    
except NameError:
    print("ERROR: 'img' variable not found!")
    print("Please load your colored image as 'img' variable first.")
    print("Example: img = cv2.imread('your_image.jpg')")
    
    # Load a sample image for demonstration
    print("\nLoading a sample image for demonstration...")
    
    # Create a sample image
    img = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
    
    plt.figure(figsize=(10, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Sample Image (Random)\nShape: {img.shape}")
    plt.axis('off')

ERROR: 'img' variable not found!
Please load your colored image as 'img' variable first.
Example: img = cv2.imread('your_image.jpg')

Loading a sample image for demonstration...


## Test JPEG Compression with Different Quality Levels

In [13]:
# Test with different quality levels
quality_levels = [90, 75, 50, 25]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, quality in enumerate(quality_levels):
    print(f"\n{'='*50}")
    print(f"Testing JPEG compression with quality = {quality}")
    print(f"{'='*50}")
    
    # Compress the image
    metadata = jpeg_compress(img, quality=quality, chroma_subsample_ratio='4:2:0')
    
    # Decompress to get reconstructed image
    reconstructed = jpeg_decompress(metadata)
    
    # Calculate metrics
    mse = np.mean((img - reconstructed) ** 2)
    psnr_value = calculate_psnr(img, reconstructed)
    compression_ratio = calculate_compression_ratio(img, metadata)
    
    # Display results
    axes[idx].imshow(cv2.cvtColor(reconstructed, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f"Quality: {quality}\nPSNR: {psnr_value:.2f} dB\nCompression: {compression_ratio:.2f}:1")
    axes[idx].axis('off')
    
    print(f"\nResults for quality {quality}:")
    print(f"  - MSE: {mse:.2f}")
    print(f"  - PSNR: {psnr_value:.2f} dB")
    print(f"  - Compression Ratio: {compression_ratio:.2f}:1")

plt.tight_layout()
plt.show()


Testing JPEG compression with quality = 90
Starting JPEG compression with quality=90, chroma subsampling=4:2:0
Step 1: Converting RGB to YCbCr...
Step 2: Applying chroma subsampling...


ValueError: all input arrays must have the same shape

## Visualize Each Step of JPEG Compression

In [ ]:
def visualize_jpeg_steps(img: np.ndarray, quality: int = 75):
    """
    Visualize each step of the JPEG compression process
    """
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    # Step 0: Original RGB Image
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("1. Original RGB Image")
    axes[0].axis('off')
    
    # Step 1: RGB to YCbCr
    img_ycbcr = rgb_to_ycbcr(img)
    axes[1].imshow(img_ycbcr[:, :, 0].astype(np.uint8), cmap='gray')
    axes[1].set_title("2. Y Channel (Luminance)")
    axes[1].axis('off')
    
    axes[2].imshow(img_ycbcr[:, :, 1].astype(np.uint8), cmap='gray')
    axes[2].set_title("3. Cb Channel (Blue Difference)")
    axes[2].axis('off')
    
    axes[3].imshow(img_ycbcr[:, :, 2].astype(np.uint8), cmap='gray')
    axes[3].set_title("4. Cr Channel (Red Difference)")
    axes[3].axis('off')
    
    # Step 2: Chroma Subsampling
    img_subsampled, _ = chroma_subsample(img_ycbcr, '4:2:0')
    axes[4].imshow(img_subsampled[:, :, 1].astype(np.uint8), cmap='gray')
    axes[4].set_title("5. Cb Channel (Subsampled 4:2:0)")
    axes[4].axis('off')
    
    # Step 3: 8x8 Blocks (visualize on Y channel)
    y_channel = img_ycbcr[:, :, 0].astype(np.uint8)
    h, w = y_channel.shape
    
    # Draw grid lines
    block_viz = y_channel.copy()
    for i in range(0, h, 8):
        cv2.line(block_viz, (0, i), (w, i), (255, 0, 0), 1)
    for j in range(0, w, 8):
        cv2.line(block_viz, (j, 0), (j, h), (255, 0, 0), 1)
    
    axes[5].imshow(block_viz, cmap='gray')
    axes[5].set_title("6. Partition into 8×8 Blocks")
    axes[5].axis('off')
    
    # Step 4: DCT Coefficients (show one block)
    blocks = split_into_blocks(img_subsampled)
    dct_blocks = apply_dct_to_blocks(blocks)
    
    # Take the first block of Y channel
    dct_block = dct_blocks[0, 0]
    axes[6].imshow(np.log1p(np.abs(dct_block)), cmap='hot')
    axes[6].set_title("7. DCT Coefficients (log scale)")
    axes[6].axis('off')
    
    # Step 5: Quantization
    quantized_blocks, _ = quantize_blocks(dct_blocks, quality)
    quant_block = quantized_blocks[0, 0]
    axes[7].imshow(np.abs(quant_block), cmap='hot')
    axes[7].set_title(f"8. Quantized (Q={quality})")
    axes[7].axis('off')
    
    # Step 6: Reconstructed Image
    metadata = jpeg_compress(img, quality=quality, chroma_subsample_ratio='4:2:0')
    reconstructed = jpeg_decompress(metadata)
    axes[8].imshow(cv2.cvtColor(reconstructed, cv2.COLOR_BGR2RGB))
    axes[8].set_title("9. Reconstructed Image")
    axes[8].axis('off')
    
    # Step 7: Difference Image
    diff = cv2.absdiff(img, reconstructed)
    axes[9].imshow(diff, cmap='hot')
    axes[9].set_title("10. Difference (Original - Reconstructed)")
    axes[9].axis('off')
    
    # Step 8: Quality Metrics
    axes[10].axis('off')
    mse = np.mean((img - reconstructed) ** 2)
    psnr_value = calculate_psnr(img, reconstructed)
    compression_ratio = calculate_compression_ratio(img, metadata)
    
    metrics_text = f"Quality Metrics:\n"
    metrics_text += f"MSE: {mse:.2f}\n"
    metrics_text += f"PSNR: {psnr_value:.2f} dB\n"
    metrics_text += f"Compression Ratio: {compression_ratio:.2f}:1\n"
    metrics_text += f"Quality Setting: {quality}"
    
    axes[10].text(0.1, 0.5, metrics_text, fontsize=12, 
                  verticalalignment='center', transform=axes[10].transAxes)
    axes[10].set_title("11. Compression Metrics")
    
    # Step 9: Side-by-side comparison
    comparison = np.hstack([cv2.cvtColor(img, cv2.COLOR_BGR2RGB), 
                            cv2.cvtColor(reconstructed, cv2.COLOR_BGR2RGB)])
    axes[11].imshow(comparison)
    axes[11].set_title("12. Original vs Reconstructed")
    axes[11].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize the JPEG compression steps
print("Visualizing JPEG compression steps...")
visualize_jpeg_steps(img, quality=75)

## Compare Different Chroma Subsampling Ratios

In [ ]:
# Test different chroma subsampling ratios
chroma_ratios = ['4:4:4', '4:2:2', '4:2:0']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, ratio in enumerate(chroma_ratios):
    print(f"\nTesting with chroma subsampling ratio: {ratio}")
    
    # Compress with current ratio
    metadata = jpeg_compress(img, quality=75, chroma_subsample_ratio=ratio)
    
    # Decompress
    reconstructed = jpeg_decompress(metadata)
    
    # Calculate metrics
    psnr_value = calculate_psnr(img, reconstructed)
    compression_ratio = calculate_compression_ratio(img, metadata)
    
    # Display
    axes[idx].imshow(cv2.cvtColor(reconstructed, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f"Chroma: {ratio}\nPSNR: {psnr_value:.2f} dB\nCompression: {compression_ratio:.2f}:1")
    axes[idx].axis('off')
    
    print(f"  - PSNR: {psnr_value:.2f} dB")
    print(f"  - Compression Ratio: {compression_ratio:.2f}:1")

plt.tight_layout()
plt.show()

## Summary and Export Functions

In [ ]:
def jpeg_summary(img: np.ndarray, quality: int = 75, chroma_ratio: str = '4:2:0'):
    """
    Provide a comprehensive summary of JPEG compression results
    """
    print("="*60)
    print("JPEG COMPRESSION SUMMARY")
    print("="*60)
    
    # Original image stats
    print(f"\nOriginal Image:")
    print(f"  - Dimensions: {img.shape[1]} x {img.shape[0]} pixels")
    print(f"  - Channels: {img.shape[2]} (RGB)")
    print(f"  - Size: {img.size * img.itemsize / 1024:.2f} KB")
    
    # Compression parameters
    print(f"\nCompression Parameters:")
    print(f"  - Quality: {quality}/100")
    print(f"  - Chroma Subsampling: {chroma_ratio}")
    print(f"  - Block Size: 8x8")
    
    # Perform compression
    metadata = jpeg_compress(img, quality=quality, chroma_subsample_ratio=chroma_ratio)
    reconstructed = jpeg_decompress(metadata)
    
    # Calculate metrics
    mse = np.mean((img - reconstructed) ** 2)
    psnr_value = calculate_psnr(img, reconstructed)
    compression_ratio = calculate_compression_ratio(img, metadata)
    
    print(f"\nCompression Results:")
    print(f"  - MSE: {mse:.2f}")
    print(f"  - PSNR: {psnr_value:.2f} dB")
    print(f"  - Compression Ratio: {compression_ratio:.2f}:1")
    
    # Display comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Original
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original Image\n{img.shape[1]}x{img.shape[0]}")
    axes[0].axis('off')
    
    # Reconstructed
    axes[1].imshow(cv2.cvtColor(reconstructed, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"JPEG Compressed\nQuality: {quality}, Chroma: {chroma_ratio}\nPSNR: {psnr_value:.2f} dB")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return metadata, reconstructed

# Run summary with default parameters
print("\n" + "="*60)
print("FINAL COMPRESSION TEST")
print("="*60)

metadata, reconstructed_img = jpeg_summary(img, quality=75, chroma_ratio='4:2:0')